# Large-Scale Portfolio Briefs Generation

## Step 1: Load Company Identifiers from CSV File

This first step reads the CSV file containing company information and extracts all unique company identifiers (RP_ENTITY_ID). These identifiers are used to specify which companies to generate briefs for.

**What it does:**
- Opens the CSV file with company data
- Finds the column containing company IDs
- Removes any empty or duplicate entries
- Creates a clean list of company identifiers for processing

**Output:** A list of unique company IDs that will be used in subsequent steps.


In [ ]:
**Note:** All required dependencies (pandas, requests, xlsxwriter, ipython, jupyterlab) should be installed from `requirements.txt` before running this notebook. See the README for installation instructions.

Audited 1 package in 23ms


In [ ]:
# Read CSV and produce comma-separated RP_ENTITY_ID string
import pandas as pd
import json

df = pd.read_csv("static/data/US_Top1000.csv", dtype=str)

col = next((c for c in df.columns if c.strip().upper() == "RP_ENTITY_ID"), None)
if col is None:
    raise ValueError("RP_ENTITY_ID column not found in static/data/US_Top1000.csv")

ids = (
    df[col]
    .astype(str)
    .str.strip()
    .replace({"": None})
    .dropna()
    .drop_duplicates()
    .tolist()
)

print(len(ids))
#print(json.dumps(ids))
# rp_entity_ids_csv now contains the comma-separated IDs

1001


## Step 2: Define Research Questions (Topics)

This step defines the specific questions that will be answered for each company in the briefing reports. These questions guide what information is gathered and summarized.

**What it does:**
- Sets up a list of research questions that will be asked about each company
- Each question is a template that will be customized with the company name
- These topics focus on key business areas: earnings, financial performance, strategic changes, contracts, and product developments

**Note:** You can modify these questions to focus on different aspects of company information based on your needs.


In [ ]:
# Build request payload (adjust dates/topics as needed)
TOPICS = [
    "What notable changes in {company}'s financial performance metrics have been reported recently?",
    "Has {company} revised its financial or operational guidance for upcoming periods?",
    "What significant strategic initiatives or business pivots has {company} announced recently?",
    "What significant contract wins, losses, or renewals has {company} recently announced?",
    "What significant new product launches or pipeline developments has {company} announced?"
]



## Step 3: Configure Batch Processing and API Settings

This step sets up the configuration for generating briefs, including how many companies to process at once and the date range for the reports.

**What it does:**
- **Batch Size:** Determines how many companies are processed together (20 companies per batch for this example)
- **Company Selection:** Selects the first 100 companies from the list (can be changed to process all companies)
- **Date Range:** Sets the time period for the briefing (start and end dates)
- **API Configuration:** Sets up authentication and the API endpoint URL, if required
- **Processing Options:** Configures how the system prioritizes information (freshness, source ranking, novelty detection)

**Key Settings:**
- `BATCH_SIZE`: Number of companies processed per request (recommended: 50 for production, max: 100)
- `report_start_date` and `report_end_date`: The time window for gathering information
- `novelty`: Whether to filter for only new or unique information
- `source_rank_boost` and `freshness_boost`: Control how sources are prioritized


In [11]:
import os
import json
import requests
import pandas as pd

# Batch size, go with 50 for prd use case, max supported is 100
BATCH_SIZE = 500

# Take first 100
companies = ids

#for prd use case use 
#companies = ids

print("Using", len(companies), "companies")
#print(json.dumps(companies))

# service uses APIKeyQuery named `token` per your OpenAPI; set env var API_TOKEN (or TOKEN/API_KEY)
token = os.environ.get("API_TOKEN") or os.environ.get("TOKEN") or os.environ.get("API_KEY")
params = {"token": token} if token else {}

payload = {
    "companies": companies,
    "report_start_date": "2025-10-27",
    "report_end_date": "2025-11-03",
    "novelty": True,
    "sources": None,
    "topics": TOPICS,
    "source_rank_boost": 10,
    "freshness_boost": 8,
    "disable_introduction": True,

}

# API call settings
# Update this to the actual create endpoint
API_URL = "http://localhost:8000/briefs/create"



Using 1001 companies


## Step 4: Set Output File Names

This step defines where the results will be saved. Two files are created:
- **Request Summary File:** Contains metadata about each batch request (status, timestamps, entity counts)
- **Combined Report File:** Contains the actual briefing data with all company reports and source information

**What it does:**
- Sets the filenames for saving the briefing results
- These files will be created automatically when the batch processing completes


In [12]:
BRIEF_SUMMARY_FILE = "briefs_request2_summaries1000.json"
BRIEF_REPORT_FILE = "combined_briefs2_report1000.json"

## Step 5: Process Companies in Batches and Generate Briefs

This is the main processing step that generates briefing reports for all companies. It works by sending requests to the API in batches and waiting for each batch to complete before moving to the next.

**What it does:**
1. **Splits companies into batches** to avoid overwhelming the API
2. **Submits each batch** to the briefing service
3. **Monitors progress** by checking the status of each request
4. **Waits for completion** (up to 10 minutes per batch)
5. **Collects results** from all batches into a single combined report
6. **Saves the results** to JSON files for later use

**Key Features:**
- **Error Handling:** If a batch fails, it records the error and continues with the next batch
- **Status Polling:** Checks every 10 seconds to see if a batch is complete
- **Automatic Merging:** Combines all batch results into one unified report
- **Progress Tracking:** Shows which batch is being processed and when it completes

**Output:** Two JSON files containing all the briefing data and request summaries.


In [ ]:
import time
import copy
import json
import traceback
import requests
from datetime import datetime 

# Get the current date and time
current_datetime = datetime.now()

# Print the full date and time
print(f"Batch Starting date and time: {current_datetime}")

def _status_url_for(request_id: str) -> str:
    # Update this to the actual status endpoint
    status_url = f"http://localhost:8000/briefs/status/{request_id}"
    return status_url

combined_entity_reports = []
combined_source_metadata = {}
request_summaries = {}

for start in range(0, len(companies), BATCH_SIZE):
    batch = companies[start:start + BATCH_SIZE]
    payload_batch = copy.deepcopy(payload)
    payload_batch["companies"] = batch

    try:
        print(f"Submitting batch {start + 1}-{start + len(batch)} ({len(batch)} entities)...")
        resp = requests.post(API_URL, params=params, json=payload_batch, timeout=180)
        resp.raise_for_status()
        create_resp = resp.json()
    except Exception as e:
        print("Create request failed for batch starting at", start, ":", e)
        traceback.print_exc()
        # store failure summary with no request id
        request_summaries[f"batch_{start}"] = {
            "start_date": payload_batch.get("report_start_date"),
            "end_date": payload_batch.get("report_end_date"),
            "logs": getattr(e, "args", str(e)),
            "report_title": None,
            "watchlist_id": None,
            "status": "create_failed"
        }
        continue

    request_id = create_resp.get("request_id")
    # capture immediate create-level logs/title if present
    immediate_report = create_resp.get("report", {}) or {}
    request_summaries[request_id or f"batch_{start}"] = {
        "start_date": payload_batch.get("report_start_date"),
        "end_date": payload_batch.get("report_end_date"),
        "logs": create_resp.get("logs") or immediate_report.get("logs"),
        "report_title": immediate_report.get("report_title") or create_resp.get("report_title"),
        "watchlist_id": immediate_report.get("watchlist_id") or create_resp.get("watchlist_id"),
        "status": "submitted"
    }

    # If no request_id, maybe synchronous response contained the report already
    if not request_id and immediate_report:
        ers = immediate_report.get("entity_reports", []) or []
        sm = immediate_report.get("source_metadata", {}) or {}
        combined_entity_reports.extend(ers)
        combined_source_metadata.update(sm)
        request_summaries[request_id or f"batch_{start}"]["status"] = "completed_sync"
        print(f"Batch {start}-{start+len(batch)} returned sync report with {len(ers)} entities.")
        continue

    # Poll status until complete/failed or timeout
    status_url = _status_url_for(request_id)
    timeout_seconds = 600  # total wait per batch
    poll_interval = 10
    waited = 0
    final_status_resp = None
    while waited < timeout_seconds:
        try:
            status_resp = requests.get(status_url, params=params, timeout=60)
            status_resp.raise_for_status()
            sjson = status_resp.json()
            status = sjson.get("status") or sjson.get("state") or ""
            if status and status.lower() in ("completed", "done", "success"):
                final_status_resp = sjson
                request_summaries[request_id]["status"] = "completed"
                break
            if status and status.lower() in ("failed", "error"):
                final_status_resp = sjson
                request_summaries[request_id]["status"] = "failed"
                break
            # otherwise still processing
        except Exception as e:
            print("Status check error:", e)
        time.sleep(poll_interval)
        waited += poll_interval

    if not final_status_resp:
        print(f"Timeout waiting for request {request_id}; proceeding to next batch.")
        request_summaries[request_id]["status"] = "timeout"
        continue

    # extract report data if present
    report = final_status_resp.get("report", {}) or final_status_resp
    entity_reports_chunk = report.get("entity_reports", []) or []
    source_meta_chunk = report.get("source_metadata", {}) or {}

    # merge
    combined_entity_reports.extend(entity_reports_chunk)
    # prefer existing keys (do not overwrite) to preserve first-seen metadata
    for k, v in source_meta_chunk.items():
        if k not in combined_source_metadata:
            combined_source_metadata[k] = v

    # update summary fields with final report metadata
    request_summaries[request_id].update({
        "logs": final_status_resp.get("logs") or request_summaries[request_id].get("logs"),
        "report_title": report.get("report_title") or request_summaries[request_id].get("report_title"),
        "watchlist_id": report.get("watchlist_id") or request_summaries[request_id].get("watchlist_id"),
        "entity_count": len(entity_reports_chunk),
        "completed_at": final_status_resp.get("completed_at") or final_status_resp.get("ts")
    })

    print(f"Batch {start + 1}-{start + len(batch)} completed: {len(entity_reports_chunk)} entities added.")

# final combined report
combined_report = {
    "entity_reports": combined_entity_reports,
    "source_metadata": combined_source_metadata
}

# persist results
with open(BRIEF_REPORT_FILE, "w", encoding="utf-8") as f:
    json.dump(combined_report, f, ensure_ascii=False, indent=2)

with open(BRIEF_SUMMARY_FILE, "w", encoding="utf-8") as f:
    json.dump(request_summaries, f, ensure_ascii=False, indent=2)

print(f"Accumulated {len(combined_entity_reports)} entity_reports and {len(combined_source_metadata)} source_metadata entries across {len(request_summaries)} requests.")

# Get the current date and time
current_datetime = datetime.now()

# Print the full date and time
print(f"Batch Completion date and time: {current_datetime}")

Batch Starting date and time: 2025-11-06 23:46:33.901263
Submitting batch 1-500 (500 entities)...


## At this stage we should have briefing of selected companies

## Step 7: Set Up Display Functions for Notebook Viewing (Reference Purpose Only)

This step defines helper functions that format and display the briefing reports in a readable way within the Jupyter notebook. These functions are used later to show the results.

**What it does:**
- **`render_source_reference()`:** Formats source information (news articles, reports) with links and metadata
- **`present_entity_report()`:** Creates a nicely formatted display of a company's briefing report with:
  - Company name, sector, industry, and country
  - Numbered bullet points summarizing key information
  - Source links for each bullet point
  - Summary statistics

**Note:** These functions are defined here but used in the next step to display results.


In [28]:
# For Presentation on Notebook 
from IPython.display import display, Markdown, HTML
import pandas as pd
import json
from pathlib import Path
from pprint import pprint
from typing import Tuple, Dict, Any, Optional
import html


def render_source_reference(
    source_id: str,
    source_metadata: Optional[Dict[str, Any]],
    show_highlights: bool = True,
    snippet_length: int = 300
) -> Tuple[str, Dict[str, Any]]:
    """
    Return (markdown_str, metadata_dict) for a given source id using the provided source_metadata map.
    - markdown_str: ready to display in a Jupyter cell via display(Markdown(...))
    - metadata_dict: the raw meta dict (for programmatic use)
    Behavior follows the selected lines: uses source_name/headline/url from the meta if present.
    """
    def _truncate(s: Optional[str], n: int) -> str:
        if not s:
            return ""
        return s if len(s) <= n else s[:n].rsplit(" ", 1)[0] + "…"

    if not source_metadata:
        md = f"`{source_id}` — (no source_metadata provided)"
        return md, {}

    meta = source_metadata.get(source_id) or {}
    if not meta:
        md = f"`{source_id}` — (not found in source_metadata)"
        return md, {}

    # chosen display fields (mirrors your selected snippet)
    name_or_headline = meta.get("source_name") or meta.get("headline") or source_id
    url = meta.get("url")
    ts = meta.get("ts")
    source_key = meta.get("source_key")
    text = meta.get("text")
    highlights = meta.get("highlights", [])

    # escape to avoid accidental HTML injection when rendering
    safe_name = html.escape(name_or_headline)
    safe_ts = html.escape(str(ts)) if ts else ""
    safe_key = html.escape(str(source_key)) if source_key else ""
    safe_snippet = html.escape(_truncate(text, snippet_length))

    # build markdown
    link_part = f"[{safe_name}]({html.escape(url)})" if url else f"**{safe_name}**"
    meta_parts = []
    if safe_ts:
        meta_parts.append(f"`{safe_ts}`")
    if safe_key:
        meta_parts.append(f"`{safe_key}`")
    meta_line = " • ".join(meta_parts)
    md_lines = [f"{link_part}  \n{meta_line}" if meta_line else f"{link_part}"]

    if safe_snippet:
        md_lines.append(f"\n> {safe_snippet}\n")

    if show_highlights and highlights:
        # highlights expected as list of {pnum:int, snum:int} or plain strings
        hl_lines = []
        for h in highlights[:6]:  # limit shown highlights
            if isinstance(h, dict):
                hl_lines.append(f"- paragraph {h.get('pnum')}, sentence {h.get('snum')}")
            else:
                hl_lines.append(f"- {html.escape(str(h))}")
        md_lines.append("**Highlights:**\n" + "\n".join(hl_lines))

    markdown = "\n\n".join(md_lines)
    return markdown, meta, link_part

def present_entity_report(entity: dict, source_metadata: dict | None = None, top_n: int | None = None):
    """
    Nicely render a single entity report for a financial analyst in a Jupyter notebook.
    - entity: the JSON object you showed (keys: entity_id, entity_info, content)
    - source_metadata: optional dict mapping source_id -> metadata (url, source_name, headline)
    - top_n: limit number of bullet points shown
    """
    ei = entity.get("entity_info", {})
    name = ei.get("name", "Unknown")
    eid = ei.get("id") or entity.get("entity_id") or "N/A"
    ticker = ei.get("ticker", "")
    sector = ei.get("sector", "—")
    industry = ei.get("industry", "—")
    country = ei.get("country", "—")
    webpage = ei.get("webpage")

    header = f"## {name}  ({eid})\n**Sector:** {sector}  •  **Industry:** {industry}  •  **Country:** {country}\n"
    if webpage:
        header += f"[Website]({webpage})\n"
    display(Markdown(header))

    bullets = entity.get("content", []) or []
    if not bullets:
        display(Markdown("_No bullet points found for this entity._"))
        return

    # Summary metrics
    num_bullets = len(bullets)
    # gather source counts
    all_srcs = []
    for b in bullets:
        all_srcs.extend(b.get("sources", []))
    src_counts = pd.Series(all_srcs).value_counts()
    top_sources = src_counts.index.tolist()[:5]
    summary_md = f"**Bullet points:** {num_bullets}  •  **Top sources (ids):** {', '.join(top_sources) if top_sources else 'None'}\n"
    display(Markdown(summary_md))

    # Show bullets (numbered) with source links if metadata provided
    limit = top_n if top_n is not None else num_bullets
    for i, b in enumerate(bullets[:limit], start=1):
        text = b.get("bullet_point", "").strip()
        srcs = b.get("sources", []) or []
        # resolve sources to friendly links/names if metadata available
        resolved = []
        markdowns = []
        for s in srcs:
            if source_metadata and s in source_metadata:
                meta = source_metadata[s]
                name_or_headline = meta.get("source_name") or meta.get("headline") or s
                url = meta.get("url")
                if url:
                    resolved.append(f"[{name_or_headline}]({url})")
                else:
                    resolved.append(f"{name_or_headline} ({s})")
            else:
                resolved.append(s)

            markdown, meta, linkpart = render_source_reference(s, source_metadata)
            markdowns.append(linkpart)

        src_line = ", ".join(resolved) if resolved else "None"
        updated_src_line = ", ".join(markdowns) if markdowns else "None"
        display(Markdown(f"{i}. {text}\n\n**Sources:** {updated_src_line}\n"))

        
    # Provide a small table for quick export / analysis
    df = pd.DataFrame([{"bullet": b.get("bullet_point", ""), "sources": b.get("sources", [])} for b in bullets])
    display(Markdown("**Raw table (for copy/export):**"))
    display(df.head(limit))


## Step 8: Load Saved Briefing Report

This step loads the briefing report that was saved during batch processing. The report contains all company briefings and their source information.

**What it does:**
- Opens the saved JSON file containing the combined briefing report
- Extracts the company reports and source metadata
- Makes the data available for display or export in subsequent steps

**Output:** 
- `entities`: List of all company briefing reports
- `source_metadata`: Dictionary mapping source IDs to source information (URLs, headlines, publication dates)


In [29]:
#read entities and source_metadata from save file 
import json
from pathlib import Path
from pprint import pprint

REPORT_PATH = Path(BRIEF_REPORT_FILE)

if not REPORT_PATH.exists():
    raise FileNotFoundError(f"{REPORT_PATH} not found. Run the batching cell that writes combined_briefs_report.json first.")

with REPORT_PATH.open("r", encoding="utf-8") as f:
    combined = json.load(f)

entities = combined.get("entity_reports", []) or []
source_metadata = combined.get("source_metadata", {}) or {}

print(f"Loaded combined report: {len(entities)} entities, {len(source_metadata)} source_metadata entries.")
# optional quick inspect
if entities:
    print("First entity keys:", list(entities[0].keys()))
if source_metadata:
    print("Sample source id:", next(iter(source_metadata.keys())))

Loaded combined report: 890 entities, 2704 source_metadata entries.
First entity keys: ['entity_id', 'entity_info', 'content']
Sample source id: 517C81647FF87DFC53F302552BD93F14-2


## Step 9: Display Sample Reports in Notebook (Reference Purpose Only)

This step displays a preview of the briefing reports directly in the notebook. It shows the first 5 companies with their top 5 bullet points each.

**What it does:**
- Takes the first 5 companies from the loaded report
- Formats each company's briefing with:
  - Company information (name, sector, industry)
  - Key bullet points summarizing important information
  - Clickable source links for each bullet point
- Displays everything in a clean, readable format

**Purpose:** Allows you to review the briefing reports immediately in the notebook before exporting to other formats.


In [30]:
# Now we can present the report in a notebook
reportable_entities = entities[:5] # picking first 5 entities for presentation
for ent in reportable_entities:
    present_entity_report(ent, source_metadata=source_metadata, top_n=5)

## Berkshire Hathaway Inc.  (168A5D)
**Sector:** Financials  •  **Industry:** Reinsurance  •  **Country:** US
[Website](http://www.berkshirehathaway.com)


**Bullet points:** 5  •  **Top sources (ids):** E244A18E54B2ADC6552E211A0EEB6797-3, C040BD630E17808994CD02C470152AE3-185, 795C09170CE2B9DE13825C6D9D526537-1, DF76C1A842FA1BFF807B224DA21C7419-5, F088020E5250068B5E92A5F9738298E4-3


1. **Berkshire Hathaway Inc.** (BRK.A) reports a significant 200% increase in insurance underwriting profit for Q3 2025, reaching $2.37 billion, attributed to a mild natural disaster season and improved operational efficiency.

**Sources:** [Benzinga](https://www.benzinga.com/node/48578414?utm_campaign=partner_feed&amp;utm_medium=feed&amp;utm_source=ravenpack)


2. The company announces a definitive agreement to acquire Occidental Petroleum's OxyChem chemicals business for $9.7 billion, marking its largest acquisition since 2022, expected to close in Q4 2025 pending regulatory approvals.

**Sources:** [Quartr Reports](https://files.quartr.com/reports/acc98-2025-11-01-01-42-54.pdf?ref=UmF2ZW5QYWNr)


3. Berkshire's operating profit rose 34% to $13.49 billion in Q3 2025, driven by strong performance in its insurance segment, while net income increased 17% to $30.8 billion, despite a decline in net investment income due to lower interest rates.

**Sources:** [The Times Of India](https://timesofindia.indiatimes.com/business/international-business/berkshire-hathaway-q3-results-profit-jumps-17-to-30-8-bn-as-buffett-readies-exit-greg-abel-set-to-take-charge-amid-381-bn-cash-pile/articleshow/125018421.cms)


...

3. Ongoing productivity improvements and restructuring efforts are key strategies for Hubbell to address rising material costs and inflation, with management noting that pricing and productivity actions have successfully offset these costs in Q3.

**Sources:** [Edgar SEC Filings](https://www.sec.gov/Archives/edgar/data/48898/000162828025046924/hubb-20250930.htm)


4. Hubbell's segment unification strategy is driving growth in key vertical markets, particularly in data centers, with strong performance expected to continue into Q4, supported by operational efficiencies and margin expansion.

**Sources:** [Benzinga](https://www.benzinga.com/node/48457271?utm_campaign=partner_feed&amp;utm_medium=feed&amp;utm_source=ravenpack)


5. The company achieved 8% organic growth in its Utility Solutions segment during Q3, with strong demand in T&D markets as utilities invest in infrastructure to meet increasing energy demands.

**Sources:** [Benzinga](https://www.benzinga.com/node/48457271?utm_campaign=partner_feed&amp;utm_medium=feed&amp;utm_source=ravenpack)


**Raw table (for copy/export):**

,bullet,sources
0,**Hubbell Inc.** (HUBB) reports a 12% increase...,[93796F3FBF3E232FDF8D157700E21D65-15]
1,The company anticipates the recent acquisition...,[E8FA93880B6E0CC237B03301AED75D04-9]
2,Ongoing productivity improvements and restruct...,[C13DC90340CE8FD122ABA489631702D3-125]
3,Hubbell's segment unification strategy is driv...,[81D914F0A07F37222022CB3C976F1EA7-2]
4,The company achieved 8% organic growth in its ...,[81D914F0A07F37222022CB3C976F1EA7-2]


## Step 10: Export Briefing Report to Excel (Reference Purpose Only)

This final step converts the briefing reports into an Excel spreadsheet format that can be easily shared, analyzed, or imported into other tools.

**What it does:**
- Creates a structured table with one row per bullet point
- Includes company information (name, sector, industry, country, website) on the first row for each company
- Lists bullet points with their associated sources
- Adds clickable hyperlinks to the source URLs in the Excel file
- Saves everything to an Excel file (.xlsx format)

**Output:** An Excel file that can be opened in Microsoft Excel, Google Sheets, or any spreadsheet application. Each row contains a bullet point, and the source column includes clickable links to the original articles or reports.


In [31]:
# Export report to Excel 

import pandas as pd
import json
from IPython.display import display

OUT_XLSX = "entities_bullets_1000.xlsx"

source_map = source_metadata

rows = []
link_meta = []  # parallel list of lists of urls (or None) for each row

for e1 in entities:
    # print (e1)
    ei = e1.get("entity_info", {}) or {}
    name = ei.get("name") or ei.get("id") or e1.get("entity_id", "")
    sectors = ei.get("sector", "")
    industry = ei.get("industry", "")
    country = ei.get("country", "")
    website = ei.get("webpage", "") or ei.get("web_site", "")

    bullets = e1.get("content", []) or []
    if not bullets:
        rows.append({
            "entity name": name,
            "sectors": sectors,
            "industry": industry,
            "country": country,
            "website": website,
            "bulletpoint": "",
            "source": ""
        })
        link_meta.append([])  # no links
        continue

    for i, b in enumerate(bullets):
        bp = b.get("bullet_point", "").strip()
        srcs = b.get("sources", []) or []

        resolved_displays = []
        resolved_urls = []
        for s in srcs:
            meta = source_map.get(s) or {}
            headline = meta.get("headline") or meta.get("source_name")
            url = meta.get("url")
            display_text = headline if headline else s
            resolved_displays.append(display_text)
            resolved_urls.append(url)  # may be None

        source_field = "; ".join(resolved_displays)
        first_url = next((u for u in resolved_urls if u), None)  # first available URL (or None)

        if i == 0:
            rows.append({
                "entity name": name,
                "sectors": sectors,
                "industry": industry,
                "country": country,
                "website": website,
                "bulletpoint": bp,
                "source": source_field
            })
        else:
            rows.append({
                "entity name": "",
                "sectors": "",
                "industry": "",
                "country": "",
                "website": "",
                "bulletpoint": bp,
                "source": source_field
            })

        link_meta.append([first_url])  # store first url (or [None])

df_out = pd.DataFrame(rows, columns=[
    "entity name", "sectors", "industry", "country", "website", "bulletpoint", "source"
])

# Write to Excel with first source as hyperlink (cell displays all source texts, link opens first URL)

with pd.ExcelWriter(OUT_XLSX, engine="xlsxwriter") as writer:
    df_out.to_excel(writer, index=False, sheet_name="Briefs")
    workbook = writer.book
    worksheet = writer.sheets["Briefs"]

    # find source column index
    src_col = df_out.columns.get_loc("source")

    # rows in sheet start at 1 (0 is header)
    for r_idx, lm in enumerate(link_meta, start=1):
        first_url = lm[0] if lm else None
        if first_url:
            display_text = df_out.iloc[r_idx - 1]["source"] or first_url
            # write_url will display the provided string but link to first_url
            worksheet.write_url(r_idx, src_col, first_url, string=display_text)

print(f"Written {len(df_out)} rows to {OUT_XLSX}")
display(df_out.head(40))



Written 2476 rows to entities_bullets_1000.xlsx


,entity name,sectors,industry,country,website,bulletpoint,source
0,Berkshire Hathaway Inc.,Financials,Reinsurance,US,http://www.berkshirehathaway.com,**Berkshire Hathaway Inc.** (BRK.A) reports a ...,Berkshire's Big Q3 Fueled By Over 200% Underwr...
1,,,,,,The company announces a definitive agreement t...,Berkshire Hathaway Inc: Q3 2025 Earnings Call ...
2,,,,,,Berkshire's operating profit rose 34% to $13.4...,Berkshire Hathaway Q3 results: Profit jumps 17...
3,,,,,,The company maintains a record cash reserve of...,Berkshire Hathaway reports record $382B reserv...
4,,,,,,As Warren Buffett prepares to step down as CEO...,Berkshire Hathaway sees profit jump as Warren ...
5,Skyworks Solutions Inc.,Technology,Semiconductors,US,http://www.skyworksinc.com,**Skyworks Solutions Inc.** (SWKS) announces a...,"Nasdaq 100 Hits 26,000, Gold Falls Below $4,00..."
6,,,,,,The company anticipates at least $500 million ...,Skyworks And Qorvo Announce $22B Merger To For...
7,,,,,,Skyworks has received upgrades from multiple a...,Piper Sandler upgrades Skyworks to Overweight ...
8,,,,,,The merger is expected to be immediately accre...,Skyworks and Qorvo to Combine to Create $22 Bi...
9,American Electric Power Co. Inc.,Utilities,Conventional Electricity,US,http://www.aep.com,The company is advancing a $72 billion capital...,"American Electric Power Co., Inc.: Q3 2025 Ear..."
